In [1]:
from transformers import pipeline
from transformers import MarianMTModel, MarianTokenizer
from evaluate import load

In [2]:
metric = load("sacrebleu")

In [8]:
#!pip install --upgrade transformers

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.0 MB 1.1 MB/s eta 0:00:09
   ---- ----------------------------------- 1.0/10.0 MB 1.3 MB/s eta 0:00:08
   ----- ---------------------------------- 1.3/10.0 MB 1.4 MB/s eta 0:00:07
   ------ --------------------------------- 1.6/10.0 MB 1.3 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/10.0 MB 1.3 MB/s eta 0:00:07
   -------- ------------------------------- 2.1/10.0 MB 1.4 MB/s eta 0:00:06
   --------- ------------------------------ 2.4/10.0 MB 1.3 MB/s eta 0:00:06
   ---------- ----------------------------- 2.6/10.0 MB 1.3 MB/s eta 0:00:06
   ----------- ---------------------------- 2.9/10.0 MB 1.3 MB/s eta 0:00:06
   ------------ --------------------------- 3.1/10.0 MB 1.2 MB/s eta 0:00:06
   ------------- ---

In [9]:
#!pip install torch

In [3]:
from transformers import pipeline

# Load translation pipeline
translator = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr")

# Sample text for translation
text = "Hello, how are you?"
translated_text = translator(text)

print("Translated text:", translated_text[0]['translation_text'])


pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

C:\Users\smita\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\smita\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-fr. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

C:\Users\smita\anaconda3\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


Translated text: Bonjour, comment allez-vous ?


In [4]:
from evaluate import load

# Load SacreBLEU metric
bleu = load("sacrebleu")

# Define reference and translated text
reference = [["Bonjour, comment allez-vous?"]]  # List of lists (because multiple references can be used)
prediction = ["Bonjour, comment vas-tu?"]

# Compute BLEU score
results = bleu.compute(predictions=prediction, references=reference)
print("BLEU Score:", results["score"])


BLEU Score: 42.7287006396234


In [5]:
# Sample text for translation
text = "Bye,Good night"
translated_text = translator(text)

print("Translated text:", translated_text[0]['translation_text'])


Translated text: Au revoir, bonne nuit.


In [1]:
import pandas as pd
import numpy as np
import string
import re
import random
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import word2vec, FastText
import gensim.downloader
from sklearn.decomposition import PCA

#device = torch.device('cude' if torch.cuda.is_available() else "cpu")

In [2]:
from datasets import load_dataset

# Load the English-French translation dataset
raw_datasets = load_dataset("kde4", lang1="en", lang2="fr")

# Display dataset structure
print(raw_datasets)


DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 210173
    })
})


In [3]:
print(f"Size of Labelled pair Data   : {raw_datasets['train'].num_rows}")

Size of Labelled pair Data   : 210173


# **Train and Test Split**

In [4]:
split_datasets = raw_datasets["train"].train_test_split(train_size=0.9, seed=20)

#rename the "test" key to "validation"

split_datasets["validation"] = split_datasets.pop("test")

In [5]:
#let’s take a look at one element of the dataset

split_datasets["train"][10:18:2]["translation"]

[{'en': 'Text Cursor Movement', 'fr': 'Mouvements du curseur de texte'},
 {'en': '2004-09-15 3.10.00', 'fr': '2004-09-15 3.10.00'},
 {'en': 'Reload the namespaces from the server. This overwrites any changes.',
  'fr': 'Recharger les espaces de noms depuis le serveur. Cette action écrasera toutes les modifications effectuées.'},
 {'en': 'Credit Card Tracker', 'fr': 'Traqueur de carte de créditName'}]

In [6]:
#let’s take a look at one element of the dataset

split_datasets["validation"][10]["translation"]

{'en': 'Read from Valgrind process failed.',
 'fr': 'Impossible de lire depuis le processus Valgrind.'}

## **Load pre-trained Model**

In [7]:
#!pip install sacremoses

In [8]:
#!pip install --upgrade transformers

In [11]:
# !pip install tf-keras

  Using cached tf_keras-2.18.0-py3-none-any.whl.metadata (1.6 kB)
Using cached tf_keras-2.18.0-py3-none-any.whl (1.7 MB)


In [14]:
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

In [15]:
import tf_keras as keras

from transformers import pipeline

#Model Name
model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"

#Load Model
translator = pipeline("translation", model=model_checkpoint)

Device set to use cpu


## **Translation from pretrained_model**

In [16]:
tmp_data = split_datasets["train"][172]["translation"]
tmp_translation = translator(tmp_data['en'])

print(f"Original English Text  :  `{tmp_data['en']}`")
print(tmp_translation)
print(f"Original French Text :  `{tmp_data['fr']}`")

Original English Text  :  `Unable to import %1 using the OFX importer plugin. This file is not the correct format.`
[{'translation_text': "Impossible d'importer %1 en utilisant le plugin d'importateur OFX. Ce fichier n'est pas le bon format."}]
Original French Text :  `Impossible d'importer %1 en utilisant le module d'extension d'importation OFX. Ce fichier n'a pas un format correct.`


## **Load Tokenizers**

In [17]:
from transformers import AutoTokenizer

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="tf")

In [30]:
print("preprocessing one sample looks like this \n")

en_sentence = split_datasets["train"][1]["translation"]["en"]
fr_sentence = split_datasets["train"][1]["translation"]["fr"]


#as_target_tokenizer() will set the tokenizer in the output language.
inputs = tokenizer(en_sentence)
with tokenizer.as_target_tokenizer():
    targets = tokenizer(fr_sentence)


wrong_targets = tokenizer(fr_sentence)
print(tokenizer.convert_ids_to_tokens(wrong_targets["input_ids"]))
print(tokenizer.convert_ids_to_tokens(targets["input_ids"]))

preprocessing one sample looks like this 

['▁Par', '▁dé', 'f', 'aut', ',', '▁dé', 've', 'lop', 'per', '▁les', '▁fil', 's', '▁de', '▁discussion', '</s>']
['▁Par', '▁défaut', ',', '▁développer', '▁les', '▁fils', '▁de', '▁discussion', '</s>']


## **Preprocessing**
- formatting data which transformers library understand.

In [18]:
max_input_length = 128
max_target_length = 128


def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # Set up the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [19]:
split_datasets["train"].column_names

['id', 'translation']

In [20]:
tokenized_datasets = split_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=split_datasets["train"].column_names,
)

## **Model Initliazation**

- `DataCollatorForSeq2Seq` pads following to the maximum length encountered in the labels.:
  - Input IDs,
  - attention mask,  
  - decoder_input_ids,
  - labels

In [22]:
from transformers import DataCollatorForSeq2Seq
from transformers import TFAutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

#model = TFAutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, from_pt=True)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, return_tensors="tf")


In [37]:
#!pip install tensorflow

  Using cached tensorflow-2.18.0-cp312-cp312-win_amd64.whl.metadata (3.3 kB)
Using cached tensorflow-2.18.0-cp312-cp312-win_amd64.whl (7.5 kB)


In [29]:
from transformers import DataCollatorForSeq2Seq
from transformers import TFAutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = TFAutoModelForSeq2SeqLM.from_pretrained(model_checkpoint, from_pt=True)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, return_tensors="tf")

All PyTorch model weights were used when initializing TFMarianMTModel.

All the weights of TFMarianMTModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMarianMTModel for predictions without further training.


In [30]:
# Lets see content of data_collator

batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
batch.keys()

dict_keys(['input_ids', 'attention_mask', 'labels', 'decoder_input_ids'])

In [24]:
# Lsts see how decoder input is mapped to id
batch["decoder_input_ids"]

tensor([[59513,   577,  5891,     2,  3184,    16,  2542,     5,  1710,     0,
         59513, 59513, 59513, 59513, 59513, 59513],
        [59513,  1211,     3,    49,  9409,  1211,     3, 29140,   817,  3124,
           817,   550,  7032,  5821,  7907, 12649]])

In [25]:
tf_train_dataset = tokenized_datasets["train"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "labels"],
    collate_fn=data_collator,
    shuffle=True,
    batch_size=32,
)
tf_eval_dataset = tokenized_datasets["validation"].to_tf_dataset(
    columns=["input_ids", "attention_mask", "labels"],
    collate_fn=data_collator,
    shuffle=False,
    batch_size=16,
)

C:\Users\smita\anaconda3\Lib\site-packages\transformers\data\data_collator.py:740: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


## **Model Compilation**
- Define optimizer
- learning rate
- number of epochs


In [31]:
from transformers import create_optimizer
import tensorflow as tf

# The number of training steps is the number of samples in the dataset, divided by the batch size then multiplied
# by the total number of epochs. Note that the tf_train_dataset here is a batched tf.data.Dataset,
# not the original Hugging Face Dataset, so its len() is already num_samples // batch_size.
num_epochs = 3
num_train_steps = len(tf_train_dataset) * num_epochs

optimizer, schedule = create_optimizer(
    init_lr=5e-5,
    num_warmup_steps=0,
    num_train_steps=num_train_steps,
    weight_decay_rate=0.01,
)
model.compile(optimizer=optimizer)

## **Model Training**

In [ ]:
model.fit(
    tf_train_dataset,
    validation_data=tf_eval_dataset,
    epochs=num_epochs,
)
model.save_pretrained('./trans-en-to-fr')

Epoch 1/3
 552/5912 [=>............................] - ETA: 84:48:57 - loss: 1.3361

## **Model Inferencing**

In [ ]:
model = TFAutoModelForSeq2SeqLM.from_pretrained('./trans-en-to-fr')

pipe = pipeline(task='translation',  
                model=model,
                tokenizer=tokenizer)


## **How to measure Translation quality?**

### **BLUE Score**

- BLEU is a metric to quantify effectiveness of an Machine Translation (MT).
- It stands for BiLingual Evaluation Understudy
- It solves the problem of different human translation references by different annotators when comparing to machine generated translation.

In [ ]:
from evaluate import load

# Load SacreBLEU metric
bleu = load("sacrebleu")

In [ ]:
import numpy as np
def compute_metrics(model):
    all_preds = []
    all_labels = []

    # Genearte sample dataset into tf_datset format from validation data
    sampled_dataset = tokenized_datasets["validation"].shuffle().select(range(200))
    tf_generate_dataset = sampled_dataset.to_tf_dataset(
        columns=["input_ids", "attention_mask", "labels"],
        collate_fn=data_collator,
        shuffle=False,
        batch_size=4,
    )

    #Generate traslation
    for batch in tf_generate_dataset:

        # predictions
        predictions = model.generate(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]
        )

        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        labels = batch["labels"].numpy()

        # removing padding pad_id = -100
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        # Ids To text
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds = [pred.strip() for pred in decoded_preds]
        decoded_labels = [[label.strip()] for label in decoded_labels]
        all_preds.extend(decoded_preds)
        all_labels.extend(decoded_labels)

    result = metric.compute(predictions=all_preds, references=all_labels)
    return {"bleu": result["score"]}

In [ ]:
print(compute_metrics(model))